In [ ]:
import optuna
from optuna import Trial
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import KFold

In [ ]:

# -------------------------
# Быстрые настройки
# -------------------------

TARGET = "Price"
N_TRIALS = 20              # вместо 50 → быстрее в 3 раза
N_FOLDS = 3                # вместо 5 → быстрее ещё в 1.6 раза
RANDOM_SEED = 42

# -------------------------
# Загрузка данных
# -------------------------


if "Id" in df.columns:
    df = df.drop(columns=["Id"])

if df["LifeSquare"].isna().sum() > 0:
    df["LifeSquare"] = df["LifeSquare"].fillna(df["LifeSquare"].median())

# Категориальные в str
cat_cols = [c for c in df.columns if str(df[c].dtype) in ("category", "object")]
if "DistrictId" not in cat_cols:
    cat_cols.append("DistrictId")

for c in cat_cols:
    df[c] = df[c].astype(str)

features = [c for c in df.columns if c != TARGET]
X = df[features]
y = df[TARGET].values

cat_feature_indices = [features.index(c) for c in cat_cols if c in features]

# -------------------------
# Optuna Objective
# -------------------------
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 800, 2500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05),
        "depth": trial.suggest_int("depth", 4, 9),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 12.0),
        "random_strength": trial.suggest_float("random_strength", 0.0, 3.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
        "one_hot_max_size": trial.suggest_int("one_hot_max_size", 30, 120),
        "loss_function": "RMSE",
        "random_seed": RANDOM_SEED,
        "verbose": False,
        "allow_writing_files": False
    }

    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    maes = []
    r2s = []

    for tr_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        train_pool = Pool(X_tr, y_tr, cat_features=cat_feature_indices)
        val_pool = Pool(X_val, y_val, cat_features=cat_feature_indices)

        model = CatBoostRegressor(**params)

        model.fit(
            train_pool, 
            eval_set=val_pool,
            early_stopping_rounds=80,   # тоже ускоряет
            use_best_model=True
        )

        pred = model.predict(val_pool)
        maes.append(mean_absolute_error(y_val, pred))
        r2s.append(r2_score(y_val, pred))

    trial.set_user_attr("r2_mean", float(np.mean(r2s)))
    return float(np.mean(maes))

# -------------------------
# Optuna study
# -------------------------
study = optuna.create_study(
    direction="minimize",
    sampler=TPESampler(seed=RANDOM_SEED),
    pruner=MedianPruner(n_warmup_steps=5)
)

print("Запуск подборa гиперпараметров...")
study.optimize(objective, n_trials=N_TRIALS)

print("\nЛучшие параметры:")
print(study.best_trial.params)
print("MAE (CV):", study.best_value)
print("R2 (CV):", study.best_trial.user_attrs["r2_mean"])

# -------------------------
# Финальная модель на всех данных
# -------------------------
best_params = study.best_trial.params
best_params.update({
    "loss_function": "RMSE",
    "random_seed": RANDOM_SEED,
    "verbose": 200,
    "allow_writing_files": False
})

final_pool = Pool(X, y, cat_features=cat_feature_indices)
final_model = CatBoostRegressor(**best_params)
final_model.fit(final_pool)

final_model.save_model("catboost_price_fast_optuna.cbm")

print("\nФинальная модель обучена и сохранена.")